# Applicazione di ELIta al corpus r/Italia — keyword *notizie*

Questo notebook applica il lessico ELIta (originale e versioni ricalcolate) ai commenti raccolti da r/Italia con keyword **notizie**.

## Import e configurazione

In [1]:
import pandas as pd
import numpy as np
import emoji
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import silhouette_score
from pathlib import Path

CORPUS_CSV   = Path('corpus_Italia_notizie.csv')
TOKENS_CSV   = Path('tokens_Italia_notizie.csv')
ELITA_CSV    = Path('../Fase1/ELIta_INTENSITY_Matrix.csv')
ALPHA_02_CSV = Path('../Fase2/output_csv/elita_recalculated_0_2.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')
ALPHA_08_CSV = Path('../Fase2/output_csv/elita_recalculated_0_8.csv')
OUTPUT_DIR   = Path('output_confronto')
OUTPUT_DIR.mkdir(exist_ok=True)

BASIC_EMOTIONS = ['gioia','tristezza','rabbia','paura','disgusto','fiducia','sorpresa','aspettativa']
EMOTION_COLORS = {
    'gioia':'#FDD835','tristezza':'#1E88E5','rabbia':'#E53935','paura':'#43A047',
    'disgusto':'#8E24AA','fiducia':'#81C784','sorpresa':'#039BE5',
    'aspettativa':'#FB8C00','neutrale':'#9E9E9E',
}
SEVEN_EMOTIONS  = ['gioia','tristezza','rabbia','paura','disgusto','fiducia','sorpresa']
POSITIVE = {'gioia','fiducia','sorpresa','aspettativa'}
NEGATIVE = {'tristezza','rabbia','paura','disgusto'}

print('Configurazione caricata.')

Configurazione caricata.


## Caricamento corpus, token e matrici ELIta

In [2]:
df_corpus = pd.read_csv(CORPUS_CSV)
df_tokens = pd.read_csv(TOKENS_CSV)
df_tokens['lemma'] = df_tokens['lemma'].astype(str).str.lower().str.strip()
df_tokens['pos']   = df_tokens['pos'].astype(str).str.upper().str.strip()
print('Corpus:', len(df_corpus), 'documenti |', 'Token:', len(df_tokens))
print('  post:', (df_corpus['type'] == 'post').sum(), '| commenti:', (df_corpus['type'] == 'comment').sum())

Corpus: 2360 documenti | Token: 80463
  post: 200 | commenti: 2160


In [3]:
df_matrix = pd.read_csv(ELITA_CSV, index_col=0)

def is_not_emoji(text):
    return emoji.emoji_count(str(text)) == 0

df_elita_orig = df_matrix[df_matrix.index.map(is_not_emoji)][BASIC_EMOTIONS].fillna(0)
df_elita_orig.index = df_elita_orig.index.astype(str).str.lower().str.strip()

def load_recalc(path):
    df = pd.read_csv(path, index_col=0)
    df.index = df.index.astype(str).str.lower().str.strip()
    return df[BASIC_EMOTIONS].fillna(0)

MATRICES = {
    'Originale (α=0)' : df_elita_orig,
    'Ibrido (α=0.2)'  : load_recalc(ALPHA_02_CSV),
    'Ibrido (α=0.5)'  : load_recalc(ALPHA_05_CSV),
    'Ibrido (α=0.8)'  : load_recalc(ALPHA_08_CSV),
}
print('Matrici:', list(MATRICES.keys()))

Matrici: ['Originale (α=0)', 'Ibrido (α=0.2)', 'Ibrido (α=0.5)', 'Ibrido (α=0.8)']


## Funzione base e prima analisi (raw)

Per ogni commento sommiamo i vettori emotivi di tutti i lemmi ADJ+NOUN+VERB trovati in ELIta.
Nessun filtro, nessuna normalizzazione.

In [4]:
def detect_emotions(df_corpus, df_tokens, df_elita, emotions=None):
    if emotions is None:
        emotions = BASIC_EMOTIONS
    pos_filter = {'ADJ','NOUN','VERB'}
    df_f = df_tokens[df_tokens['pos'].isin(pos_filter)].copy()
    eidx = set(df_elita.index)
    tok  = df_f.groupby('doc_id')['lemma'].apply(list).to_dict()
    results = []
    for _, row in df_corpus.iterrows():
        did   = row['doc_id']
        lemmi = tok.get(did, [])
        sc = {e: 0.0 for e in emotions}
        found = 0
        for lemma in lemmi:
            if lemma in eidx:
                found += 1
                for e in emotions:
                    sc[e] += df_elita.loc[lemma, e]
        results.append({'doc_id': did, 'type': row.get('type', ''), 'n_tokens_matched': found, **sc,
            'dominant_emotion': max(sc, key=sc.get) if found > 0 else 'neutrale'})
    return pd.DataFrame(results)

df_raw     = detect_emotions(df_corpus, df_tokens, df_elita_orig)
counts_raw = df_raw['dominant_emotion'].value_counts()
total      = len(df_raw)

df_raw.to_csv(OUTPUT_DIR / 'notizie_emotion_results_raw.csv', index=False)
print(f'Salvato: {OUTPUT_DIR / "notizie_emotion_results_raw.csv"}')

print('Distribuzione emozione dominante — raw:')
for e in BASIC_EMOTIONS + ['neutrale']:
    n = counts_raw.get(e,0)
    print('{:<15s} {:>4d} ({:>4.1f}%) {}'.format(e, n, n/total*100, '█'*int(n/total*40)))

Salvato: output_confronto/notizie_emotion_results_raw.csv
Distribuzione emozione dominante — raw:
gioia            319 (13.5%) █████
tristezza         83 ( 3.5%) █
rabbia            96 ( 4.1%) █
paura            112 ( 4.7%) █
disgusto          12 ( 0.5%) 
fiducia          104 ( 4.4%) █
sorpresa          72 ( 3.1%) █
aspettativa     1382 (58.6%) ███████████████████████
neutrale         180 ( 7.6%) ███


In [5]:
fig = go.Figure()
for e in BASIC_EMOTIONS + ['neutrale']:
    n = counts_raw.get(e, 0)
    fig.add_trace(go.Bar(name=e, x=[e], y=[round(n/total*100,1)],
        marker_color=EMOTION_COLORS.get(e,'#999'),
        text=['{:.0f}%'.format(n/total*100)], textposition='outside'))
fig.update_layout(title='Distribuzione emozione dominante — analisi raw',
                  barmode='group', height=450, showlegend=False)
fig.show()

**Osservazione**: aspettativa domina massicciamente (~60%).
Come documentato in ItEm (Pollacci 2015), alcune emozioni producono coseni più alti diventando "catalizzanti". Il primo passo è capire se questo bias è strutturale o semantico.
Seguiamo l'approccio di ItEm: ripetiamo l'analisi escludendo aspettativa.

## Passo 1 — Rimozione di aspettativa (corpus_7emo)

Come `corpus_sei_emo` in ItEm (che escludeva fiducia e attese), escludiamo aspettativa
per vedere la struttura emotiva sottostante. Non è il metodo finale — serve a esplorare.

In [6]:
df_7emo    = detect_emotions(df_corpus, df_tokens, df_elita_orig, emotions=SEVEN_EMOTIONS)
counts_7   = df_7emo['dominant_emotion'].value_counts()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Raw (8 emozioni)','Senza aspettativa (7 emozioni)'],
    horizontal_spacing=0.1)
for col_idx,(df_r,emos) in enumerate([(df_raw,BASIC_EMOTIONS),(df_7emo,SEVEN_EMOTIONS)],start=1):
    counts = df_r['dominant_emotion'].value_counts()
    for e in emos+['neutrale']:
        n = counts.get(e,0)
        fig.add_trace(go.Bar(name=e, x=[e], y=[round(n/total*100,1)],
            marker_color=EMOTION_COLORS.get(e,'#999'),
            showlegend=(col_idx==1), legendgroup=e,
            text=['{:.0f}%'.format(n/total*100)], textposition='outside'),
            row=1, col=col_idx)
fig.update_layout(title='Raw vs corpus_7emo (senza aspettativa)',
                  barmode='group', height=500)
fig.show()

**Osservazione**: senza aspettativa emergono sorpresa e fiducia come dominanti.
Questa soluzione però è artificiosa: rimuove informazione invece di correggere il bias.
L'approccio corretto secondo ItEm è la normalizzazione.

## Passo 2 — Corpus_mean di ItEm (Formula 3.5)

In ItEm il corpus_mean normalizza **dopo** aver accumulato i punteggi grezzi di ogni
documento. La logica è:

```
Passo A  S_e(d)      = Σ_{w ∈ d} score(e, w)       ← score grezzo del doc (già il nostro "raw")
Passo B  μ_e         = (1/N) Σ_d S_e(d)             ← media corpus per emozione e
Passo C  S_e_norm(d) = S_e(d) / μ_e                  ← score normalizzato del documento
```

L'obiettivo: se un'emozione ha un valore medio alto su tutto il corpus (bias sistematico),
dividerla per la sua media la riporta su una scala comparabile con le altre emozioni.
Aspettativa ha μ ≈ 12.7, le altre emozioni hanno μ tra 4 e 10 — dopo la normalizzazione
aspettativa viene ridimensionata proporzionalmente di più.

In [7]:
def apply_corpus_mean_norm(df_results, emotions=None):
    """corpus_mean di ItEm (Formula 3.5):
    per ogni emozione divide il punteggio di ogni documento per la media di quell'emozione sull'intero corpus.
    Input : DataFrame con colonne per emozione (output di detect_emotions).
    Output: DataFrame con score normalizzati e dominant_emotion ricalcolato."""
    if emotions is None:
        emotions = BASIC_EMOTIONS
    mu     = df_results[emotions].mean()   # μ_e per ogni emozione
    df_norm = df_results.copy()
    for e in emotions:
        if mu[e] > 0:
            df_norm[e] = df_results[e] / mu[e]
    # ricomputa emozione dominante sugli score normalizzati
    df_norm['dominant_emotion'] = df_norm[emotions].apply(
        lambda r: r.idxmax() if r.sum() > 0 else 'neutrale', axis=1)
    return df_norm, mu

# Applica corpus_mean agli score raw
df_corpus_mean, mu_raw = apply_corpus_mean_norm(df_raw)
counts_cm = df_corpus_mean['dominant_emotion'].value_counts()

print('Medie corpus per emozione (μ_e — score raw):')
for e in BASIC_EMOTIONS:
    print('  {:<15s}: {:.4f}'.format(e, mu_raw[e]))
print()
print('Distribuzione — corpus_mean ItEm (Formula 3.5):')
for e in BASIC_EMOTIONS + ['neutrale']:
    n  = counts_cm.get(e, 0)
    n0 = counts_raw.get(e, 0)
    print('{:<15s} {:>4d} ({:>4.1f}%)  [{:+d} vs raw]'.format(e, n, n/total*100, n-n0))

Medie corpus per emozione (μ_e — score raw):
  gioia          : 3.2912
  tristezza      : 2.1441
  rabbia         : 2.0553
  paura          : 2.3724
  disgusto       : 1.2846
  fiducia        : 3.1073
  sorpresa       : 2.6088
  aspettativa    : 4.2274

Distribuzione — corpus_mean ItEm (Formula 3.5):
gioia            384 (16.3%)  [+65 vs raw]
tristezza        245 (10.4%)  [+162 vs raw]
rabbia           174 ( 7.4%)  [+78 vs raw]
paura            195 ( 8.3%)  [+83 vs raw]
disgusto         422 (17.9%)  [+410 vs raw]
fiducia          291 (12.3%)  [+187 vs raw]
sorpresa         290 (12.3%)  [+218 vs raw]
aspettativa      179 ( 7.6%)  [-1203 vs raw]
neutrale         180 ( 7.6%)  [+0 vs raw]


In [8]:
# Applica corpus_mean a tutte le versioni ibride
results_cm_ibridi    = {}
results_raw_versions = {}
for vname, df_e in MATRICES.items():
    df_raw_v = detect_emotions(df_corpus, df_tokens, df_e)
    results_raw_versions[vname] = df_raw_v
    df_cm_v, mu_v = apply_corpus_mean_norm(df_raw_v)
    results_cm_ibridi[vname] = df_cm_v

# Visualizzazione 2×2: raw | corpus_mean orig | corpus_mean α0.5 | corpus_mean α0.8
panel_cm = [
    ('Raw (score originali)',            df_raw),
    ('corpus_mean ItEm (score orig)',    results_cm_ibridi['Originale (α=0)']),
    ('corpus_mean ItEm (ibrido α=0.5)', results_cm_ibridi['Ibrido (α=0.5)']),
    ('corpus_mean ItEm (ibrido α=0.8)', results_cm_ibridi['Ibrido (α=0.8)']),
]

fig = make_subplots(rows=2, cols=2,
    subplot_titles=[l for l, _ in panel_cm],
    vertical_spacing=0.18, horizontal_spacing=0.08)
positions = [(1,1),(1,2),(2,1),(2,2)]

for idx, (label, df_r) in enumerate(panel_cm):
    counts = df_r['dominant_emotion'].value_counts()
    r, c = positions[idx]
    for e in BASIC_EMOTIONS + ['neutrale']:
        n = counts.get(e, 0)
        fig.add_trace(go.Bar(
            name=e, x=[e], y=[round(n/total*100, 1)],
            marker_color=EMOTION_COLORS.get(e, '#999'),
            showlegend=(idx == 0), legendgroup=e,
            text=['{:.0f}%'.format(n/total*100)], textposition='outside'),
            row=r, col=c)

fig.update_layout(
    title='Corpus_mean ItEm (Formula 3.5): confronto versioni',
    barmode='group', height=700)
fig.show()

# Tabella riassuntiva: effetto corpus_mean su aspettativa
print('\nEffetto corpus_mean ItEm su aspettativa:')
print('{:<30s} | {:>8s} | {:>8s}'.format('Versione', 'Raw', 'corpus_mean'))
print('-' * 53)
for vname in MATRICES:
    c_raw = results_raw_versions[vname]['dominant_emotion'].value_counts()
    c_cm  = results_cm_ibridi[vname]['dominant_emotion'].value_counts()
    print('{:<30s} | {:>7.1f}% | {:>7.1f}%'.format(
        vname,
        c_raw.get('aspettativa', 0)/total*100,
        c_cm.get('aspettativa',  0)/total*100))


Effetto corpus_mean ItEm su aspettativa:
Versione                       |      Raw | corpus_mean
-----------------------------------------------------
Originale (α=0)                |    58.6% |     7.6%
Ibrido (α=0.2)                 |    59.6% |     8.3%
Ibrido (α=0.5)                 |    59.2% |     8.7%
Ibrido (α=0.8)                 |    55.3% |     9.2%


### Conclusione Passo 2 — corpus_mean ItEm su ELIta

Corpus_mean di ItEm (Formula 3.5) **riduce aspettativa** dal **60%** (raw) a **8%**.

**Meccanismo**: Dividendo per lo score raw, ogni emozione viene riportata su una scala comparabile. Le emozioni con valore medio alto nel corpus vengono penalizzate proporzionalmente alla loro diffusione.

**Questo è il metodo finale**: aggregazione grezza → corpus_mean (Formula 3.5).

## Diagnosi del bias — top driver di aspettativa

Per capire perché il raw mostra 60% di aspettativa, identifichiamo le parole che contribuiscono di più: **frequenza × score ELIta = contributo totale**.
Il corpus_mean normalizza questi contributi dividendo per lo score raw.

In [9]:
POS_FILTER = {'ADJ', 'NOUN', 'VERB'}
df_filt   = df_tokens[df_tokens['pos'].isin(POS_FILTER)].copy()
elita_idx = set(df_elita_orig.index)

matched = sorted(set(df_filt['lemma']).intersection(elita_idx))
freq    = df_filt[df_filt['lemma'].isin(matched)]['lemma'].value_counts()

freq_df = freq.reset_index()
freq_df.columns = ['lemma', 'frequenza']
er = df_elita_orig.loc[matched, BASIC_EMOTIONS].reset_index()
er.columns = ['lemma'] + BASIC_EMOTIONS
freq_df = freq_df.merge(er, on='lemma', how='left')
freq_df['word_sum']    = freq_df[BASIC_EMOTIONS].sum(axis=1)
freq_df['contrib_asp'] = freq_df['frequenza'] * freq_df['aspettativa']

print('Top 25 parole per contributo ad ASPETTATIVA (frequenza × score):')
display(freq_df.nlargest(25, 'contrib_asp')[
    ['lemma', 'frequenza', 'aspettativa', 'word_sum', 'contrib_asp']
].round(3).reset_index(drop=True))

Top 25 parole per contributo ad ASPETTATIVA (frequenza × score):


,lemma,frequenza,aspettativa,word_sum,contrib_asp
0,fare,702,0.58,1.32,407.16
1,avere,465,0.58,2.86,269.70
2,notizia,199,0.71,3.12,141.29
3,vedere,230,0.54,2.63,124.20
4,dire,282,0.38,1.93,107.16
5,anno,188,0.54,1.79,101.52
6,pensare,126,0.75,3.75,94.50
7,dare,116,0.71,2.76,82.36
8,trovare,84,0.92,4.04,77.28
9,stare,113,0.54,2.03,61.02


**Osservazione**: le prime parole (*notizia*, *fare*, *avere*, *vedere*...) sono lemmi generici o legati al topic.
`fare` da sola vale **107 punti** (702 occorrenze × 0.58) — la keyword di ricerca è il principale driver di aspettativa nel raw.

Il corpus_mean ridimensiona questi contributi: dividendo per score raw, anche le parole con alta frequenza e score elevato vengono pesate proporzionalmente al loro ruolo medio nel corpus.

## Metodo finale: corpus_mean di ItEm su tutte le versioni ELIta

Applichiamo la **corpus_mean normalisation** (Formula 3.5 di ItEm) a tutte e quattro le versioni:

```
S_e_norm(d) = S_e(d) / μ_e     dove     μ_e = mean_d(S_e(d))
```

Nessun filtraggio lessicale — il bias di aspettativa è gestito interamente dalla normalizzazione.

In [10]:
results_final = {}
for vname, df_e in MATRICES.items():
    df_raw_v = detect_emotions(df_corpus, df_tokens, df_e)
    df_r, _  = apply_corpus_mean_norm(df_raw_v)
    results_final[vname] = df_r
    matched = (df_raw_v['n_tokens_matched'] > 0).sum()
    print('{:<20s} | match: {:d}/{:d} ({:.0f}%)'.format(
          vname, matched, len(df_r), matched/len(df_r)*100))

Originale (α=0)      | match: 2180/2360 (92%)
Ibrido (α=0.2)       | match: 2180/2360 (92%)
Ibrido (α=0.5)       | match: 2180/2360 (92%)
Ibrido (α=0.8)       | match: 2180/2360 (92%)


In [11]:
fig = make_subplots(rows=2, cols=2, subplot_titles=list(MATRICES.keys()),
    vertical_spacing=0.18, horizontal_spacing=0.08)
positions = [(1,1),(1,2),(2,1),(2,2)]
for idx,(vname,df_r) in enumerate(results_final.items()):
    counts = df_r['dominant_emotion'].value_counts()
    r,c = positions[idx]
    for e in BASIC_EMOTIONS + ['neutrale']:
        n = counts.get(e,0)
        fig.add_trace(go.Bar(name=e, x=[e], y=[round(n/total*100,1)],
            marker_color=EMOTION_COLORS.get(e,'#999'),
            showlegend=(idx==0), legendgroup=e,
            text=['{:.0f}%'.format(n/total*100)], textposition='outside'),
            row=r, col=c)
fig.update_layout(
    title='Metodo finale: corpus_mean ItEm (Formula 3.5)',
    barmode='group', height=700)
fig.show()

## Confronto quantitativo: originale vs ricalcolato

Misuriamo l'impatto del ricalcolo su aspettativa e sulle altre emozioni, confrontando raw e corpus_mean.

In [12]:
# Confronto progressione: raw → corpus_mean (metodo finale)
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Raw', 'corpus_mean ItEm (metodo finale)'],
    horizontal_spacing=0.12)
for col_idx, df_r in enumerate([df_raw, df_corpus_mean], start=1):
    counts = df_r['dominant_emotion'].value_counts()
    for e in BASIC_EMOTIONS + ['neutrale']:
        n = counts.get(e, 0)
        fig.add_trace(go.Bar(name=e, x=[e[:4]], y=[round(n/total*100, 1)],
            marker_color=EMOTION_COLORS.get(e, '#999'),
            showlegend=(col_idx==1), legendgroup=e,
            text=['{:.0f}%'.format(n/total*100)], textposition='outside'),
            row=1, col=col_idx)
fig.update_layout(
    title='Progressione: Raw → corpus_mean ItEm',
    barmode='group', height=500)
fig.show()

In [13]:
# Bilancio pos/neg
print('{:<35s} | {:>10s} | {:>10s}'.format('Configurazione','Positive','Negative'))
print('-'*62)
for label, df_r in [('Raw',              df_raw),
                    ('corpus_mean ItEm', df_corpus_mean)]:
    c = df_r['dominant_emotion'].value_counts()
    pos = sum(c.get(e, 0) for e in POSITIVE)
    neg = sum(c.get(e, 0) for e in NEGATIVE)
    print('{:<35s} | {:>9.1f}% | {:>9.1f}%'.format(label, pos/total*100, neg/total*100))

Configurazione                      |   Positive |   Negative
--------------------------------------------------------------
Raw                                 |      79.5% |      12.8%
corpus_mean ItEm                    |      48.5% |      43.9%


Possiamo vedere che la versione con α=0.5 riduce il gap tra aspettativa e le altre emozioni rispetto alla versione originale.

## 10. Tabella riassuntiva

In [14]:
# Tabella A: N. documenti e token con score > 0 per emozione
df_raw_orig = results_raw_versions['Originale (α=0)']
table_A = []
for e in BASIC_EMOTIONS:
    n_doc   = int((df_raw_orig[e] > 0).sum())
    doc_ids = set(df_raw_orig[df_raw_orig[e] > 0]['doc_id'])
    n_tok   = int(df_filt[
        df_filt['doc_id'].isin(doc_ids) &
        df_filt['lemma'].isin(set(df_elita_orig.index))
    ]['lemma'].count())
    table_A.append({'Emozione': e.capitalize(),
                    'N. Documenti (score>0)': n_doc,
                    'N. Token': n_tok})
print('Tabella A — N. documenti e token per emozione (metodo finale: corpus_mean):')
display(pd.DataFrame(table_A))

Tabella A — N. documenti e token per emozione (metodo finale: corpus_mean):


,Emozione,N. Documenti (score>0),N. Token
0,Gioia,2151,23726
1,Tristezza,2142,23713
2,Rabbia,2124,23688
3,Paura,2148,23727
4,Disgusto,2087,23627
5,Fiducia,2161,23744
6,Sorpresa,2169,23751
7,Aspettativa,2169,23750


In [15]:
dom = results_final['Originale (α=0)']['dominant_emotion'].value_counts()
tot = len(results_final['Originale (α=0)'])
table_B = [{'Emozione':e.capitalize(),
            'N. Documenti dom':int(dom.get(e,0)),
            '% totale':'{:.1f}%'.format(dom.get(e,0)/tot*100)}
           for e in BASIC_EMOTIONS+['neutrale']]
print('\nTabella B — Emozione dominante (corpus_mean ItEm, Formula 3.5):')
display(pd.DataFrame(table_B))


Tabella B — Emozione dominante (corpus_mean ItEm, Formula 3.5):


,Emozione,N. Documenti dom,% totale
0,Gioia,384,16.3%
1,Tristezza,245,10.4%
2,Rabbia,174,7.4%
3,Paura,195,8.3%
4,Disgusto,422,17.9%
5,Fiducia,291,12.3%
6,Sorpresa,290,12.3%
7,Aspettativa,179,7.6%
8,Neutrale,180,7.6%


## Conclusioni

### Percorso seguito

- **Raw**: aspettativa domina (~60%). Il bias è numerico: aspettativa ha il raw score, molto più alto rispetto alle altre emozioni.
- **Rimozione aspettativa** (corpus_7emo): utile per l'esplorazione, non come metodo finale.
- **Corpus_mean ItEm** (Formula 3.5): aspettativa scende a **8%**. Divide ogni score per la media di corpus di quell'emozione — le emozioni sistematicamente alte vengono penalizzate proporzionalmente.

### Effetto corpus_mean su aspettativa|

Il corpus_mean ridimensiona il bias numerico di aspettativa senza rimuovere alcuna parola dal lessico.
Il residuo (~8%) riflette la genuina caratterizzazione semantica del dominio *notizie*.